<a href="https://colab.research.google.com/github/langchain-samples/lc-colab-workshops/blob/main/notebooks/00_setup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 00 · Setup and preflight

**Run this before the workshop — ideally a few days before, not the morning of.**

You need exactly one thing: a **LangSmith API key**. No OpenAI key, no Anthropic key, no
credit card, nothing installed on your laptop. Models are served through the LangSmith
gateway, so that single key covers the models, the tracing, and the Studio UI for all 14
lessons.

This notebook takes about 10 minutes. It walks you through creating the key, storing it
safely, and then runs five checks. Each check prints either ✅ or ❌ **with the specific fix**
— so if something is wrong, you will know exactly what to do about it.

> **Why run this early?** Check 3 can fail for reasons you cannot fix yourself, like the model
> gateway not being enabled for your workspace. That is a five-minute fix on a Tuesday and a
> ruined morning at 9:01 on workshop day.

---

## Step 1 — Get a LangSmith API key

1. Go to [smith.langchain.com](https://smith.langchain.com) and sign in (or sign up — the free
   tier is enough for this workshop).
2. Open **Settings → API Keys**.
3. Click **Create API Key**, give it a name like `workshop`, and create it.
4. **Copy it now.** LangSmith shows the key exactly once. If you lose it, delete the key and
   make a new one — there is no way to see it again.

Your key starts with `lsv2_`.

> 📸 **`00-langsmith-settings.png`** — LangSmith Settings → API Keys page, with the
> **Create API Key** button visible.
>
> *Caption:* Settings → API Keys is where the key is created.

> 📸 **`00-create-key-modal.png`** — The create-key modal with a name filled in, and the
> generated key shown with its copy button.
>
> *Caption:* Copy the key here. This is the only time it is shown.

---

## Step 2 — Store the key in Colab Secrets

1. In the **left sidebar of this notebook**, click the 🔑 key icon.
2. Click **Add new secret**.
3. Name it **exactly** `LANGSMITH_API_KEY` — capitals, underscores, no spaces.
4. Paste your key into the **Value** field.
5. **Turn on the "Notebook access" toggle.**

**Step 5 is the one everybody misses.** A secret with notebook access off behaves exactly like
a secret that was never created — the notebook simply cannot see it. If Check 1 below fails
even though you are certain you saved the key, this toggle is almost always the reason.

Also note: the name is `LANGSMITH_API_KEY`, **not** `LANGCHAIN_API_KEY`. The older name still
appears in a lot of blog posts and will not be picked up here.

> 📸 **`00-colab-secrets.png`** — Colab's Secrets panel with `LANGSMITH_API_KEY` saved and the
> **Notebook access** toggle switched ON and circled in red.
>
> *Caption:* The toggle on the right must be on. Off looks identical to "no secret at all".

### Why a secret instead of pasting the key into a cell?

Notebooks get shared, screen-shared during workshops, downloaded, and committed to GitHub. A
key pasted into a cell travels with all of those. Colab Secrets keep the value out of the
notebook file entirely, and every lesson in this course reads the key the same way.

If you are running this outside Colab (local Jupyter, VS Code), the cells below fall back to
reading `LANGSMITH_API_KEY` from your environment, or prompting for it.

---

## Step 3 — Install and run the preflight

Installing takes a minute or two. It is worth doing here rather than discovering a dependency
problem in the first lesson.

In [ ]:
# --- snippet:install v1 ---
%pip install -qq \
  "deepagents~=0.7.6" \
  "langchain~=1.3.15" \
  "langchain-openai~=1.5.1" \
  "langsmith~=0.11.0" \
  "git+https://github.com/langchain-samples/langsmith-studio-nb.git"
# --- /snippet ---

print("Install finished. If you saw a red dependency-conflict warning above, note it —")
print("Check 3 is where it would show up as a real problem.")

### The five checks

Run the next cells in order. Each prints ✅ or ❌.

**If you hit a ❌, stop and fix it before moving on** — the later checks build on the earlier
ones, so one failure tends to cascade into several. The final cell prints a single-line summary
you can paste into the workshop chat if you need help.

| Check | Proves | Needed for |
|---|---|---|
| 1 | The notebook can read your secret | Everything |
| 2 | Your key is valid | Everything |
| 3 | The gateway will serve you a model | Everything |
| 4 | Your runs are recorded in LangSmith | Lesson 01, and all of Part 2 |
| 5 | Studio can tunnel to this runtime | Lessons 01, 06, 09 (optional) |

In [ ]:
# --- Check 1 — can this notebook read your secret? ---
import os

RESULTS = []


def report(name, ok, detail="", remedy=""):
    """Record a check and print it with a fix attached when it fails."""
    RESULTS.append({"name": name, "ok": ok, "remedy": remedy})
    print(f"{'✅' if ok else '❌'} {name}")
    if detail:
        print(f"   {detail}")
    if not ok and remedy:
        print(f"   → {remedy}")
    return ok


SECRET_NAME = "LANGSMITH_API_KEY"
api_key = None

try:
    from google.colab import userdata
except ImportError:
    userdata = None

if userdata is None:
    # Not on Colab — fall back to the environment, then to a prompt.
    from getpass import getpass

    api_key = os.environ.get(SECRET_NAME) or getpass(f"{SECRET_NAME}: ")
    report("Check 1 — secret reachable", bool(api_key), "Not running on Colab; read from environment.")
else:
    try:
        api_key = userdata.get(SECRET_NAME)
        report("Check 1 — secret reachable", bool(api_key))
    except Exception as exc:
        # Colab raises different types for "no such secret" and "access not granted",
        # and the fixes are completely different. Match on the name so this keeps
        # working if the exception classes move.
        kind = type(exc).__name__
        if "NotebookAccess" in kind:
            remedy = (
                'The secret exists but this notebook cannot read it. Open the 🔑 panel and turn '
                'ON "Notebook access" for LANGSMITH_API_KEY, then re-run this cell.'
            )
        elif "SecretNotFound" in kind:
            remedy = (
                "No secret named LANGSMITH_API_KEY. Check the spelling in the 🔑 panel — "
                "LANGCHAIN_API_KEY is a different (older) name and will not be found."
            )
        else:
            remedy = f"Unexpected error ({kind}): {exc}"
        report("Check 1 — secret reachable", False, remedy=remedy)

if api_key:
    os.environ["LANGSMITH_API_KEY"] = api_key
    os.environ["LANGSMITH_TRACING"] = "true"
    os.environ["LANGSMITH_PROJECT"] = "lcw-00-setup"
    # Never print the key itself — length and last 4 are enough to spot a truncated paste.
    print(f"   key length {len(api_key)}, ends ...{api_key[-4:]}")

In [ ]:
# --- Check 2 — is the key valid, and are you pointed at the right region? ---
import requests

ENDPOINT = os.environ.get("LANGSMITH_ENDPOINT", "https://api.smith.langchain.com").rstrip("/")

try:
    response = requests.get(
        f"{ENDPOINT}/api/v1/sessions",
        params={"limit": 1},
        headers={"x-api-key": api_key},
        timeout=30,
    )
    if response.status_code == 200:
        report("Check 2 — key is valid", True, f"Authenticated against {ENDPOINT}")
    elif response.status_code in (401, 403):
        report(
            "Check 2 — key is valid",
            False,
            f"{ENDPOINT} returned {response.status_code}.",
            "Either the key is wrong/revoked, or your workspace is in the EU. For an EU "
            "workspace, run: os.environ['LANGSMITH_ENDPOINT'] = 'https://eu.api.smith.langchain.com' "
            "and re-run this cell.",
        )
    else:
        report(
            "Check 2 — key is valid",
            False,
            f"Unexpected status {response.status_code}: {response.text[:200]}",
            "Retry once. If it persists, LangSmith may be having an incident — check status.smith.langchain.com",
        )
except requests.exceptions.RequestException as exc:
    report(
        "Check 2 — key is valid",
        False,
        f"Could not reach {ENDPOINT}: {type(exc).__name__}",
        "This is usually a network/proxy problem rather than a key problem. On a corporate "
        "VPN, try again off the VPN.",
    )

In [ ]:
# --- Check 3 — will the gateway serve you a model? ---
from langchain.chat_models import init_chat_model

# Every lesson pins its model in one constant exactly like this one.
MODEL = "langsmith:openai/gpt-5.6-luna"

try:
    llm = init_chat_model(MODEL)
    answer = llm.invoke("Reply with exactly one word: ready")
    report("Check 3 — gateway serves models", True, f"{MODEL} replied: {answer.text!r}")
except Exception as exc:
    message = str(exc)
    if "gateway:invoke" in message:
        remedy = (
            "Your key authenticates (Check 2 passed) but lacks the 'gateway:invoke' permission, "
            "so it cannot call models. This happens with scoped keys. Create a new key with "
            "default permissions, or ask a workspace admin to enable the LLM gateway."
        )
    elif "model" in message.lower() and ("not found" in message.lower() or "404" in message):
        remedy = (
            f"The gateway does not recognise {MODEL}. Model names change; ask the workshop host "
            "for the current one and update the MODEL constant above."
        )
    elif "429" in message or "quota" in message.lower():
        remedy = "Rate limited or out of credits on this workspace. Check your LangSmith plan/usage."
    else:
        remedy = f"Unrecognised failure: {type(exc).__name__}. Paste this into the workshop chat."
    report("Check 3 — gateway serves models", False, message[:300], remedy)

In [ ]:
# --- Check 4 — do your runs actually reach LangSmith? ---
# Everything can "work" while nothing is being recorded. This proves the round trip:
# emit a trace, then read it back by id.
import time
import warnings

from langsmith import Client, trace

# list_runs() is deprecated in favour of an async-only replacement, which would force
# `await` into a preflight cell. It is supported until 2027; keep the sync call and hide
# the notice so a passing check prints clean output.
warnings.filterwarnings("ignore", message=".*list_runs.*deprecated.*")

try:
    client = Client()

    with trace(name="preflight-tracing-check", inputs={"check": 4}) as run_tree:
        run_tree.end(outputs={"ok": True})
        run_id = run_tree.id

    client.flush()

    run_url = None
    try:
        run_url = run_tree.get_url()
    except Exception:
        pass  # URL resolution is a nicety; ingestion below is the real test.

    found = False
    for _ in range(10):  # ingestion is asynchronous, so poll rather than assume
        if list(client.list_runs(id=[run_id])):
            found = True
            break
        time.sleep(2)

    if found:
        report("Check 4 — tracing round-trips", True, f"Run {run_id} is queryable.")
        if run_url:
            print(f"   View it: {run_url}")
    else:
        report(
            "Check 4 — tracing round-trips",
            False,
            f"Run {run_id} was sent but is not readable after ~20s.",
            "Confirm LANGSMITH_TRACING is 'true' (Check 1 sets it), then re-run this cell "
            "once — ingestion is asynchronous and occasionally lags.",
        )
except Exception as exc:
    # Never let this cell raise: the summary below has to run so you get the full picture.
    message = str(exc)
    if "403" in message or "Forbidden" in message:
        remedy = (
            "Your key can read the workspace but cannot write traces — a scoped/read-only key. "
            "Note Check 2 still passes in this state. Create a key with default permissions."
        )
    elif "401" in message:
        remedy = "Key rejected. Re-check Check 2 before this one."
    else:
        remedy = f"Unexpected {type(exc).__name__}. Paste this into the workshop chat."
    report("Check 4 — tracing round-trips", False, message[:300], remedy)

In [ ]:
# --- Check 5 (optional) — can LangSmith Studio reach this runtime? ---
# Studio runs your agent behind a public tunnel. This is the flakiest check and the least
# critical: if it fails you can still complete every lesson, you just lose the visual
# debugger. Skip it if you are short on time.
try:
    from deepagents import create_deep_agent
    from langsmith_studio_nb import start_studio

    # start_studio() serves the notebook variable named `agent` by default.
    agent = create_deep_agent(model=MODEL, system_prompt="You are a test agent.")
    session = start_studio()
    report("Check 5 — Studio tunnel (optional)", True, "Open the Studio link printed above.")
except Exception as exc:
    report(
        "Check 5 — Studio tunnel (optional)",
        False,
        f"{type(exc).__name__}: {str(exc)[:200]}",
        "Optional — lessons still work without it. Corporate networks often block the tunnel; "
        "trying off-VPN usually fixes it.",
    )

In [ ]:
# --- Summary ---
required = [r for r in RESULTS if "optional" not in r["name"]]
failed = [r for r in required if not r["ok"]]

print("=" * 62)
if not failed:
    optional_failed = [r for r in RESULTS if "optional" in r["name"] and not r["ok"]]
    print("✅ ALL CHECKS PASSED — you are ready for the workshop.")
    if optional_failed:
        print("   (Studio is unavailable, which is fine. Lessons still work.)")
else:
    print(f"❌ {len(failed)} of {len(required)} required checks FAILED:")
    for r in failed:
        print(f"   - {r['name']}")
    print()
    print("Fix the FIRST failure above and re-run from that cell — one failure usually")
    print("causes the ones after it. The troubleshooting table below covers the common cases.")
print("=" * 62)

---

## Troubleshooting

| Symptom | Cause | Fix |
|---|---|---|
| Check 1 fails, but you saved the secret | **"Notebook access" toggle is off** | 🔑 panel → toggle it on → re-run. This is the most common failure by a wide margin. |
| Check 1 fails, secret "not found" | Named `LANGCHAIN_API_KEY` | Rename it to `LANGSMITH_API_KEY`. The old name is not read. |
| Check 1 fails after a long break | Runtime was recycled | Re-run every cell from the top. Colab discards installs and variables when a runtime is reclaimed. |
| Check 2 returns 401/403 | Key revoked, mistyped, or truncated on paste | Compare the printed key length against your key. Re-copy if it differs. |
| Check 2 returns 401/403 and your workspace is in the EU | Wrong regional endpoint | `os.environ["LANGSMITH_ENDPOINT"] = "https://eu.api.smith.langchain.com"` before Check 2. |
| Check 2 cannot connect at all | Corporate proxy or VPN | Try off-VPN. This is a network problem, not a key problem. |
| Check 3 says `missing permission gateway:invoke` | The key is **scoped** and cannot call models | Create a new key with default permissions, or have an admin enable the gateway for the workspace. Checks 1 and 2 pass in this state, which is what makes it confusing. |
| Check 4 says `403 Forbidden` | Same root cause: a scoped key that can read but not write traces | Create a key with default permissions. Check 2 passes here too — reading and writing are separate permissions. |
| Red `Failed to multipart ingest runs` text appears | The background tracer could not upload, usually alongside a Check 4 failure | Not a separate problem. Fix Check 4 and this stops. |
| Check 3 says the model is unknown | Model name has been retired | Ask the host for the current name and edit the `MODEL` constant. |
| Check 4 fails but 1–3 pass | Tracing disabled, or ingestion lag | Confirm `LANGSMITH_TRACING="true"`, then re-run the cell once. |
| Check 5 fails | Tunnel blocked | Optional. Ignore it, or try off-VPN. |
| `%pip install` prints red conflict warnings | Colab ships preinstalled packages that clash | Usually harmless. It only matters if Check 3 also fails. |

### 🧠 Checkpoint

You saved the key in Colab Secrets, the name is spelled correctly, and Check 1 still fails.
What is the first thing to look at?

<details><summary>Show answer</summary>

The **"Notebook access" toggle** in the 🔑 Secrets panel.

Colab scopes each secret to the notebooks you explicitly grant access to. With the toggle off,
`userdata.get("LANGSMITH_API_KEY")` fails in a way that is indistinguishable at a glance from
the secret not existing — which sends people off re-creating a key that was fine all along.
That is why Check 1 catches the two cases separately and prints different fixes.

</details>

---

## What you just proved

| Check | Used by |
|---|---|
| Key readable + valid | Every lesson |
| Gateway serves models | Every lesson |
| Tracing round-trips | Lesson 01, and all of Part 2 (evals read your traces) |
| Studio tunnel | Lessons 01, 06, 09 — nice to have, not required |

Keep this notebook. If something breaks mid-workshop, re-running these five checks is the
fastest way to find out whether the problem is your setup or the lesson.

---

## ➡️ Next

**[01 · The whole game: your first Deep Agent](https://colab.research.google.com/github/langchain-samples/lc-colab-workshops/blob/main/notebooks/01_first_deep_agent.ipynb)**

You will build a research assistant that plans its own work, takes notes to a filesystem, and
writes a report — in about ten lines of code — and then take apart what made that possible.